## Calculade a clasificación do circuíto aplicando as normas de puntuación do regulamento.

In [7]:
import os
import hmac
import hashlib
import unicodedata
import pandas as pd
from dotenv import load_dotenv

# ==============================================================================
# 0. CONFIGURACIÓN Y REGLAS DEL REGLAMENTO (Art. 3, 4 y 9)
# ==============================================================================
load_dotenv()
CLAVE_SECRETA = os.getenv("HMAC_SECRET_KEY", "").encode("utf-8")

REGLAS_CIRCUITO = {
    "min_pruebas_clasificar": 5,         # Art. 9 (pág. 11): obligatorio terminar 5 pruebas[cite: 59]
    "max_pruebas_computables": 5,        # Art. 9 (pág. 11): se descartan los peores resultados[cite: 59]
    "columna_puntos": "puesto_por_sexo", # Probar 'puesto_por_sexo' vs 'puesto_general'[cite: 58, 59]
    "valor_inscrito": "inscrito_circuito", # Ajustar si en vuestro CSV se llama distinto
    "valor_tiempo_valido": "valido"        # Ajustar si en vuestro CSV se llama distinto (ej. 'ok')
}

# ==============================================================================
# 1. CÁLCULO PROPIO SEGÚN REGLAMENTO (sobre datos/clean/participacions.csv)
# ==============================================================================
df_part = pd.read_csv("../datos/clean/participacions.csv")

# Comprobación automática por si los valores de texto en vuestro CSV son ligeramente distintos
if REGLAS_CIRCUITO["valor_inscrito"] not in df_part["tipo_inscripcion"].unique():
    print("⚠️ Revisa tipo_inscripcion, valores disponibles:", df_part["tipo_inscripcion"].unique())
if REGLAS_CIRCUITO["valor_tiempo_valido"] not in df_part["estado_tiempo"].unique():
    print("⚠️ Revisa estado_tiempo, valores disponibles:", df_part["estado_tiempo"].unique())

# Filtrar solo inscritos al circuito y tiempos válidos[cite: 52, 58]
df_valido = df_part[
    (df_part["tipo_inscripcion"] == REGLAS_CIRCUITO["valor_inscrito"]) &
    (df_part["estado_tiempo"] == REGLAS_CIRCUITO["valor_tiempo_valido"])
].copy()

col_pts = REGLAS_CIRCUITO["columna_puntos"]
df_valido["puntos_carrera"] = pd.to_numeric(df_valido[col_pts], errors="coerce")
df_valido = df_valido.dropna(subset=["puntos_carrera"])

# Ordenar las carreras de cada persona de menor (mejor) a mayor (peor) puntuación
df_valido = df_valido.sort_values(["persoa_id", "puntos_carrera", "id_carrera"])
df_valido["orden_resultado"] = df_valido.groupby("persoa_id").cumcount() + 1

# Quedarnos con las 5 mejores puntuaciones de cada persona[cite: 59]
df_top5 = df_valido[df_valido["orden_resultado"] <= REGLAS_CIRCUITO["max_pruebas_computables"]]

# Agrupar resultados por persona
resumen = df_valido.groupby(["persoa_id", "sexo", "categoria"], as_index=False).agg(
    pruebas_finalizadas=("id_carrera", "count"),
    mejores_puestos=("puntos_carrera", lambda s: tuple(sorted(s)))
)

puntos = df_top5.groupby("persoa_id", as_index=False)["puntos_carrera"].sum().rename(
    columns={"puntos_carrera": "puntos_calc"}
)

df_propia = resumen.merge(puntos, on="persoa_id", how="inner")

# Exigir mínimo 5 pruebas terminadas[cite: 59]
df_propia = df_propia[df_propia["pruebas_finalizadas"] >= REGLAS_CIRCUITO["min_pruebas_clasificar"]].copy()

# Ordenar aplicando desempates del Art. 9:
# 1º Menor suma de puntos | 2º Más pruebas finalizadas | 3º Mejores puestos[cite: 59]
df_propia["neg_pruebas"] = -df_propia["pruebas_finalizadas"]
df_propia = df_propia.sort_values(
    by=["sexo", "puntos_calc", "neg_pruebas", "mejores_puestos"]
).drop(columns=["neg_pruebas"])

df_propia["puesto_sexo_calc"] = df_propia.groupby("sexo").cumcount() + 1
df_propia["puesto_cat_calc"] = df_propia.groupby(["sexo", "categoria"]).cumcount() + 1

print(f"✅ Clasificación propia calculada: {len(df_propia)} personas clasificadas.")

# ==============================================================================
# 2. CARGA Y PSEUDONIMIZACIÓN DE LA CLASIFICACIÓN OFICIAL (Persona 1)
# ==============================================================================
def normalizar_nombre(texto: str) -> str:
    if pd.isna(texto):
        return ""
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("utf-8")
    return " ".join(texto.upper().split())

def generar_hmac(nombre_normalizado: str) -> str:
    return hmac.new(CLAVE_SECRETA, nombre_normalizado.encode("utf-8"), hashlib.sha256).hexdigest()

# ⬇️ RELLENA AQUÍ LA RUTA Y COLUMNAS DEL FICHERO OFICIAL QUE DESCARGÓ PERSONA 1:
RUTA_OFICIAL = "../datos/interim/resultados_limpios_local.csv"  # <-- Pon aquí vuestro archivo real

if os.path.exists(RUTA_OFICIAL):
    df_oficial_raw = pd.read_csv(RUTA_OFICIAL)
    
    # <-- Cambia 'Nombre', 'Puntos' y 'Puesto' por los nombres exactos de columnas de ese CSV:
    df_oficial_raw["persoa_id"] = df_oficial_raw["nombre"].apply(normalizar_nombre).apply(generar_hmac)
    
    df_oficial = df_oficial_raw[["persoa_id", "Puntos", "Puesto"]].rename(
        columns={"Puntos": "puntos_oficial", "Puesto": "puesto_oficial"}
    )
    # Borramos de memoria la tabla con nombres reales[cite: 38, 40]
    del df_oficial_raw

    # ==========================================================================
    # 3. CRUCE Y AUDITORÍA DE DISCREPANCIAS[cite: 42]
    # ==========================================================================
    comparativa = pd.merge(df_propia, df_oficial, on="persoa_id", how="outer", indicator=True)
    comparativa["diff_puntos"] = comparativa["puntos_calc"] - comparativa["puntos_oficial"]
    comparativa["diff_puesto"] = comparativa["puesto_sexo_calc"] - comparativa["puesto_oficial"]

    print("\n--- RESUMEN DEL CRUCE ---")
    print(comparativa["_merge"].value_counts())

    discrepancias = comparativa[
        (comparativa["_merge"] != "both") |
        (comparativa["diff_puntos"].fillna(0) != 0) |
        (comparativa["diff_puesto"].fillna(0) != 0)
    ]
    print(f"\n⚠️ Discrepancias encontradas: {len(discrepancias)}")
    display(discrepancias.head(20))
else:
    print(f"ℹ️ Aún no existe {RUTA_OFICIAL}. Pide a Persona 1 el archivo descargado de la web para cruzarlo.")

✅ Clasificación propia calculada: 292 personas clasificadas.


KeyError: "['Puntos', 'Puesto'] not in index"